# 根据种子数据扩增&数据合成

In [ ]:
from importlib.metadata import version

print(f"openai version:  {version("openai")}")
print(f"python-dotenv version:  {version("python-dotenv")}")

In [ ]:
import os
import re
import pickle
import random
import time
import threading
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

from dotenv import load_dotenv
from openai import OpenAI

# 加载环境变量
load_dotenv()

random.seed(42)

# 提示词

In [ ]:
QUERY_EXPANSION_PROMPT = """
你是一个经验丰富的文字改写专家，对中国的旅游城市，交通，景点路线等非常熟悉。你的任务是根据输入的用户关于旅游的问题，扩充生成5个新的问题。

以下是一些生成新问题的要求：
1.给的用户问题只是参考，要输出完全新的问题，不能跟原有的句子意思相同或相近。
2.新的问题里面出现的地名，景点，必须是真实的，问题是符合逻辑的，请结合你的只旅游知识来进行改写，可以适当替换里面出现的地点/城市。
3.(可选)如果有必要，可以在提问里面增加一些约束，例如价格，距离，评分，交通方式，预算，途经点等。

请直接输出改写后的句子，每一个句子是一行，不要输出理由或者其他无关内容。

用户问题：{}
"""

# 定义 OpenAI Client

In [ ]:
llm_client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_BASE_URL"]
)

# 写调用逻辑

In [ ]:
def chat(prompt, max_retry=3):
    def do_chat(text):
        prompt = QUERY_EXPANSION_PROMPT.format(text)
        completions = llm_client.chat.completions.create(
            model="deepseek-v4-pro",
            messages=[
                {"role": "system", "content": "你是有用的人工智能助手。"},
                {"role": "user", "content": prompt}
            ]
        )
        return text, completions.choices[0].message.content

    while max_retry > 0:
        try:
            return do_chat(prompt)
        except Exception as e:
            max_retry -= 1
            sleep_seconds = random.randint(1, 4)
            time.sleep(sleep_seconds)
    return  

# 测试一下

In [ ]:
result = chat("五一假期从重庆到宜昌3日高铁，途经恩施，自然风光和土家族文化，带小孩，预算总6000，推荐吊脚楼。")

In [ ]:
print(result[1])

# 并发调大模型

In [ ]:
MAX_WORKERS = 50        # 并发度 50，可根据实际的大模型接口 QPS 调整

input_path = "./data/train_seed.jsonl"
output_path = "./data/train_expansion.jsonl"


queries = []
with open(input_path) as fd:
    for idx, line in enumerate(fd):
        info = json.loads(line)
        queries.append((idx, info["question"]))

expand_queries = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {idx: executor.submit(chat, query) for idx, query in queries}
    for idx in tqdm(futures):
        future = futures[idx]
        result = future.result()
        if result is None:
            continue
        query, expand = result
        expand = expand.split("\n")
        expand = [re.sub(r"\d. ", "", item) for item in expand if item.strip()]
        expand_queries.append(query)
        expand_queries.extend(expand)

# 去重
expand_queries = list(set(expand_queries))

with open(output_path, "w") as fw:
    for query in expand_queries:
        info = {"question": query}
        fw.write(json.dumps(info, ensure_ascii=False) + "\n")

In [ ]:
! cat ../data/train_expansion.jsonl | shuf -n 10